# Import Required Libraries
This block imports all necessary libraries and sets up the environment for the soil moisture model.

In [ ]:
from dataclasses import dataclass, field, fields, replace
from typing import Dict, List, Optional, Sequence
import numpy as np
import pandas as pd
import xarray as xr 
import matplotlib.pyplot as plt
import sys
from pathlib import Path
src_path = Path("/home/vm/Desktop/soil_moisture_model/src")
if str(src_path) not in sys.path : sys.path.insert(0, str(src_path))
from soil_moisture_model.soil_moisture_model import SoilWaterParams, simulate_soil_water_balance, params_as_dict

# ICOS Station Data Preparation
This block contains code for preparing and filtering ICOS station data, including plant type selection and site name extraction.

In [ ]:
icosStations = pd.read_csv("/mnt/shared/pyrealm2/ICOS_stations_processed.csv")
icosStations = icosStations[icosStations['Site type'].isin([
    'croplands','grasslands','forest','open shrublands',
    'savannas','closed shrublands','evergreen needleleaf forests',
    'evergreen broadleaf forests','deciduous needleleaf forests',
    'deciduous needleleaf forests','mixed forests'])]


# Site Data Loading and Preprocessing
This block loads site-specific data, performs LAI to biomass conversion, and prepares FLUXNET data for further analysis.

In [ ]:
# site_name = icosStations.Id.iloc[1]  # Adjust as needed
# D_monthly = xr.open_dataset(f"/mnt/shared/pyrealm2/inputData/{site_name}_weekly_final_variables.nc")
# # Simple LAI to biomass conversion: biomass (gC/m2) = LAI * 100 (common approximation)
# biomass = D_monthly.modis_lai[:,0,0].data * 100


# # Load FLUXNET data - adjust the file path as needed
# D = pd.read_csv(f"/mnt/shared/pyrealm2/ICOS_ETC_ARCHIVE/tmp_{site_name}/ICOSETC_{site_name}_FLUXNET_HH_L2.csv")
# date_col = "TIMESTAMP"  # adjust column name if needed

# # Build datetime index from FLUXNET timestamp
# site_csv_df = D.copy()
# site_csv_df[date_col] = pd.to_datetime(site_csv_df['TIMESTAMP_START'].astype(str), format="%Y%m%d%H%M")
# site_csv_df['SWC_F_MDS_1'] = site_csv_df['SWC_F_MDS_1'].replace(-9999, np.nan)

# n = len(D_monthly.time.data)
# example = pd.DataFrame({
#     "week": D_monthly.time.data,
#     "temp_c": D_monthly.temperature_celcius[:,0,0].data,
#     "precip_mm": D_monthly.precipitation_mm[:,0,0].data,
#     "biomass_gC_m2": biomass,
# })

# out = simulate_soil_water_balance(example, date_col="week", 
#                                   elevation_m = float(icosStations[icosStations['Id'] == site_name]['Elevation above sea'].values[0]) ,     
#                                   clay_pct = float(D_monthly.attrs['clay_fraction']))


In [ ]:
# # Build weekly SWC from half-hourly observations (week ending Friday to match `out["week"]`)
# swc_weekly = (
#     site_csv_df.assign(
#         week=site_csv_df["TIMESTAMP"].dt.to_period("W-FRI").dt.end_time.dt.normalize()
#     )
#     .groupby("week", as_index=False)["SWC_F_MDS_1"]
#     .mean()
# )

# # Keep modeled soil moisture and merge on week
# merged_sm = (
#     out[["week", "soil_moisture_mm"]]
#     .copy()
#     .assign(week=lambda d: pd.to_datetime(d["week"]))
#     .merge(swc_weekly, on="week", how="inner")
#     .dropna(subset=["soil_moisture_mm", "SWC_F_MDS_1"])
# )

# # Scatter plot
# ax = merged_sm.plot.scatter(
#     x="soil_moisture_mm",
#     y="SWC_F_MDS_1",
#     figsize=(7, 5),
#     alpha=0.7,
#     title=f"{site_name}: Modeled soil moisture vs observed SWC"
# )
# ax.set_xlabel("Modeled soil_moisture_mm")
# ax.set_ylabel("Observed SWC_F_MDS_1")

# Load and Display Calibration Results
This block loads the best parameter and results tables from disk and displays them for review.

In [ ]:
best_params_by_site = pd.read_parquet("/home/vm/Desktop/best_params_by_site.parquet")
print(len(best_params_by_site))
# print(best_params_by_site)
results_all_sites = pd.read_parquet("/home/vm/Desktop/results_all_sites.parquet")
# print(results_all_sites)

## Post-calibration Site-Specific Time Series Plots
This block generates time series plots for modeled and observed soil moisture, precipitation, ET, runoff, and drainage for a selected site.

In [ ]:
site = results_all_sites.index.get_level_values("site_name").unique()[43]
site_df = results_all_sites.xs(site, level="site_name").reset_index()

# 2-column layout (2x2) and tighter spacing
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 8.5))
axes = axes.ravel()

# Soil moisture: modeled vs observed (time series)
axes[0].plot(site_df["time"], site_df["soil_moisture_mm"], label="Modeled soil_moisture_mm", color="steelblue")
if "SWC_F_MDS_1" in site_df.columns:
    axes[0].plot(site_df["time"], site_df["SWC_F_MDS_1"], label="Observed SWC_F_MDS_1", color="orange", alpha=0.8)
axes[0].set_title(f"{site} — Soil Moisture")
axes[0].set_ylabel("mm / fraction")
axes[0].legend()

# Scatter: simulated vs observed soil moisture (now 2nd subplot)
ax_sc = axes[1]
if "SWC_F_MDS_1" in site_df.columns:
    comp = site_df[["soil_moisture_mm", "SWC_F_MDS_1"]].dropna()
    if len(comp) > 1:
        x = comp["soil_moisture_mm"]
        y = comp["SWC_F_MDS_1"]

        ax_sc.scatter(x, y, s=22, alpha=0.65, color="teal", edgecolor="none")

        lo = min(x.min(), y.min())
        hi = max(x.max(), y.max())
        ax_sc.plot([lo, hi], [lo, hi], "--", color="black", linewidth=1.2)
        ax_sc.set_xlim(lo, hi)
        ax_sc.set_ylim(lo, hi)

        r = np.corrcoef(x, y)[0, 1]
        rmse = np.sqrt(np.mean((x - y) ** 2))

        ax_sc.text(
            0.03, 0.97,
            f"r = {r:.3f}\nRMSE = {rmse:.2f}",
            transform=ax_sc.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray")
        )
    else:
        ax_sc.text(0.5, 0.5, "Not enough paired data for scatter", ha="center", va="center", transform=ax_sc.transAxes)
else:
    ax_sc.text(0.5, 0.5, "SWC_F_MDS_1 not available", ha="center", va="center", transform=ax_sc.transAxes)

ax_sc.set_title(f"{site} — Simulated vs Observed Soil Moisture")
ax_sc.set_xlabel("Modeled soil_moisture_mm")
ax_sc.set_ylabel("Observed SWC_F_MDS_1")

# Runoff
axes[2].plot(site_df["time"], site_df["runoff_mm"], label="Runoff (mm)", color="brown")
axes[2].set_title(f"{site} — Runoff")
axes[2].set_ylabel("mm")
axes[2].legend()

# Precipitation and ET (moved to 4th subplot)
axes[3].bar(site_df["time"], site_df["precip_mm"], label="Precip (mm)", color="cornflowerblue", alpha=0.6, width=5)
axes[3].plot(site_df["time"], site_df["et_actual_mm"], label="ET actual (mm)", color="green")
axes[3].plot(site_df["time"], site_df["et_potential_mm"], label="ET potential (mm)", color="red", linestyle="--")
axes[3].set_title(f"{site} — Precipitation & ET")
axes[3].set_ylabel("mm")
axes[3].legend()

fig.tight_layout(pad=0.7, w_pad=0.6, h_pad=0.7)
plt.show()

## Parameter Distribution Visualization
This block visualizes the distribution of optimized model parameters by plant type using boxplots.

In [ ]:
# Plot parameter distributions per plant_type from best_params_by_site
df_params = best_params_by_site.copy()

# Keep numeric columns that are actual parameters/fit values
exclude_cols = {"calib_rmse"}  # keep/remove as you prefer
param_cols = [
    c for c in df_params.select_dtypes(include=[np.number]).columns
    if c not in exclude_cols
]

plant_types = sorted(df_params["plant_type"].dropna().unique())

n = len(param_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.8 * nrows))
axes = np.array(axes).reshape(-1)

for i, p in enumerate(param_cols):
    ax = axes[i]
    data = [df_params.loc[df_params["plant_type"] == pt, p].dropna().values for pt in plant_types]
    data = [d for d in data if len(d) > 0]
    labels = [pt for pt in plant_types if not df_params.loc[df_params["plant_type"] == pt, p].dropna().empty]

    if data:
        ax.boxplot(data, tick_labels=labels, patch_artist=True)
    ax.set_title(p)
    ax.set_xlabel("plant_type")
    ax.tick_params(axis="x", rotation=30)

for j in range(n, len(axes)):
    axes[j].axis("off")

fig.suptitle("best_params_by_site: Parameter distribution by plant_type", y=1.01)
fig.tight_layout()
plt.show()

# Batch Model Simulation for All Sites
This block runs the soil moisture model for all sites, merges observed SWC, and summarizes the results.

In [ ]:
# # Use icosSites if it exists, otherwise fall back to icosStations
# sites_df = icosSites if "icosSites" in globals() else icosStations

# all_results = []
# failed_sites = []
# failed_swc = []

# for site_name in sites_df["Id"].dropna().unique():
#     try:
#         # ---------- Model inputs ----------
#         D_site = xr.open_dataset(f"/mnt/shared/pyrealm2/inputData/{site_name}_weekly_final_variables.nc")

#         site_meta = sites_df.loc[sites_df["Id"] == site_name].iloc[0]
#         elevation = float(site_meta["Elevation above sea"])
#         plant_type = site_meta["plant_type"] if "plant_type" in site_meta.index else np.nan

#         lai_to_biomass_multiplier = {
#             "tree": 600.0,
#             "shrub": 100.0,
#             "grass": 100.0,
#             "crop": 200.0,
#         }.get(str(plant_type).strip().lower(), 100.0)

#         biomass = D_site.modis_lai[:, 0, 0].data * lai_to_biomass_multiplier
#         clay_pct = float(D_site.attrs["clay_fraction"])

#         example_site = pd.DataFrame({
#             "time": D_site.time.data,
#             "temp_c": D_site.temperature_celcius[:, 0, 0].data,
#             "precip_mm": D_site.precipitation_mm[:, 0, 0].data,
#             "biomass_gC_m2": biomass,
#         })

#         out_site = simulate_soil_water_balance(
#             example_site,
#             date_col="time",
#             elevation_m=elevation,
#             clay_pct=clay_pct
#         )
#         out_site["time"] = pd.to_datetime(out_site["time"])

#         # ---------- Observed SWC from FLUXNET ----------
#         try:
#             D_flux = pd.read_csv(
#                 f"/mnt/shared/pyrealm2/ICOS_ETC_ARCHIVE/tmp_{site_name}/ICOSETC_{site_name}_FLUXNET_HH_L2.csv"
#             )
#             site_csv_df = D_flux.copy()
#             site_csv_df["TIMESTAMP"] = pd.to_datetime(
#                 site_csv_df["TIMESTAMP_START"].astype(str), format="%Y%m%d%H%M"
#             )
#             site_csv_df["SWC_F_MDS_1"] = site_csv_df["SWC_F_MDS_1"].replace(-9999, np.nan)

#             swc_weekly_site = (
#                 site_csv_df.assign(
#                     time=site_csv_df["TIMESTAMP"].dt.to_period("W-FRI").dt.end_time.dt.normalize()
#                 )
#                 .groupby("time", as_index=False)["SWC_F_MDS_1"]
#                 .mean()
#             )

#             out_site = out_site.merge(swc_weekly_site, on="time", how="left")

#         except Exception as e:
#             out_site["SWC_F_MDS_1"] = np.nan
#             failed_swc.append((site_name, str(e)))

#         out_site["site_name"] = site_name
#         out_site["plant_type"] = plant_type
#         all_results.append(out_site)

#     except Exception as e:
#         failed_sites.append((site_name, str(e)))

# results_all_sites_def = (
#     pd.concat(all_results, ignore_index=True)
#     .set_index(["time", "site_name", "plant_type"])
#     .sort_index()
# )

# print(f"Done. Successful sites: {len(all_results)} | Failed model sites: {len(failed_sites)} | Failed SWC loads: {len(failed_swc)}")
# if failed_sites:
#     print("Model failed (first 5):", failed_sites[:5])
# if failed_swc:
#     print("SWC load failed (first 5):", failed_swc[:5])


In [ ]:
# Prepare copies with keys as columns
calib = results_all_sites.reset_index() if isinstance(results_all_sites.index, pd.MultiIndex) else results_all_sites.copy()
default = results_all_sites_def.reset_index() if isinstance(results_all_sites_def.index, pd.MultiIndex) else results_all_sites_def.copy()

# Normalize key names
rename_map = {"id": "site_name", "Id": "site_name", "site_type": "plant_type"}
calib = calib.rename(columns={k: v for k, v in rename_map.items() if k in calib.columns})
default = default.rename(columns={k: v for k, v in rename_map.items() if k in default.columns})

# Ensure time alignment
if "time" in calib.columns:
    calib["time"] = pd.to_datetime(calib["time"])
if "time" in default.columns:
    default["time"] = pd.to_datetime(default["time"])

# Optional typo fix
if "plant_type" in calib.columns:
    calib["plant_type"] = calib["plant_type"].replace({"shurb": "shrub"})
if "plant_type" in default.columns:
    default["plant_type"] = default["plant_type"].replace({"shurb": "shrub"})

# Tag result source and combine
calib["results_group"] = "calib"
default["results_group"] = "def"

merged_results = pd.concat([calib, default], ignore_index=True, sort=False)

# Keep a grouped view if needed
grouped_results = merged_results.groupby("results_group")

# Scatter split into one subplot per results_group
df = merged_results.copy()

group_candidates = ["results_group", "run_group", "scenario", "config", "set_name"]
group_col = next((c for c in group_candidates if c in df.columns), None)
if group_col is None:
    raise ValueError("No group column found (expected e.g. 'def' / 'calib').")

df[group_col] = (
    df[group_col].astype(str).str.strip().str.lower().replace({
        "default": "def",
        "baseline": "def",
        "calibrated": "calib",
        "calibration": "calib",
    })
)

plot_df = df.dropna(subset=["soil_moisture_mm", "SWC_F_MDS_1"]).copy()
groups = sorted(plot_df[group_col].unique())

if len(groups) == 0:
    raise ValueError("No paired data after dropping NaNs in soil_moisture_mm and SWC_F_MDS_1.")

fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 6), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

colors = {"def": "tab:blue", "calib": "tab:orange"}
lo = min(plot_df["soil_moisture_mm"].min(), plot_df["SWC_F_MDS_1"].min())
hi = max(plot_df["soil_moisture_mm"].max(), plot_df["SWC_F_MDS_1"].max())

for ax, g in zip(axes, groups):
    d = plot_df[plot_df[group_col] == g]
    ax.scatter(
        d["soil_moisture_mm"], d["SWC_F_MDS_1"],
        s=18, alpha=0.33, color=colors.get(g, "gray")
    )
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"{group_col} = {g} (n={len(d)})")
    ax.set_xlabel("soil_moisture_mm")
    ax.set_ylabel("SWC_F_MDS_1")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

fig.suptitle("soil_moisture_mm vs SWC_F_MDS_1 by results_group", y=1.02)
plt.tight_layout()
plt.show()



In [ ]:

# Choose outputs to visualize (simulation outputs, not raw drivers)
output_cols = [
    "soil_moisture_mm",
    "relative_soil_moisture",
    "water_stress",
    "runoff_mm",
    "drainage_mm",
    "et_actual_mm",
    "et_potential_mm",
    "intercepted_mm",
    "throughfall_mm",
    "kc",
]

# Keep only columns that exist
output_cols = [c for c in output_cols if c in results_all_sites_def.columns]

# Mean weekly per plant_type for model outputs
weekly_mean = (
    results_all_sites_def[output_cols]
    .groupby(level=["time", "plant_type"])
    .mean()
    .reset_index()
)
weekly_mean["time"] = pd.to_datetime(weekly_mean["time"])

# Mean weekly per plant_type for comparison subplot
comp_df = (
    results_all_sites[["soil_moisture_mm", "SWC_F_MDS_1"]]
    .groupby(level=["time", "plant_type"])
    .mean()
    .reset_index()
    .dropna(subset=["soil_moisture_mm", "SWC_F_MDS_1"])
)

# Fixed plant type colors
color_map = {
    "crop": "orange",
    "grass": "green",
    "shrub": "blue",
    "tree": "red",
}

# Build subplot grid (+1 for comparison subplot)
n = len(output_cols) + 1
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4.2 * nrows), sharex=False)
axes = np.array(axes).reshape(-1)

# Time-series subplots
for i, col in enumerate(output_cols):
    plot_df = weekly_mean.pivot(index="time", columns="plant_type", values=col)
    plot_df = plot_df.rename(columns={"shurb": "shrub"})
    ordered_types = [pt for pt in ["crop", "grass", "shrub", "tree"] if pt in plot_df.columns]

    if ordered_types:
        plot_df[ordered_types].plot(
            ax=axes[i],
            linewidth=1.2,
            legend=False,
            color=[color_map[pt] for pt in ordered_types],
        )

    axes[i].set_title(f"Mean Weekly {col}")
    axes[i].set_xlabel("Time")
    axes[i].set_ylabel(col)

# Comparison subplot: modeled soil moisture vs observed SWC
ax_cmp = axes[len(output_cols)]
for pt in ["crop", "grass", "shrub", "tree"]:
    d = comp_df[comp_df["plant_type"] == pt]
    if not d.empty:
        ax_cmp.scatter(
            d["soil_moisture_mm"],
            d["SWC_F_MDS_1"],
            s=16,
            alpha=0.55,
            color=color_map[pt],
            label=pt
        )

# 1:1 reference line
x = comp_df["soil_moisture_mm"]
y = comp_df["SWC_F_MDS_1"]
lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax_cmp.plot([lo, hi], [lo, hi], "--", color="black", linewidth=1.2, label="1:1")
ax_cmp.set_xlim(lo, hi)
ax_cmp.set_ylim(lo, hi)

# Metrics (overall): r, RMSE, bias (modeled - observed)
if len(comp_df) > 1:
    r = np.corrcoef(x, y)[0, 1]
    rmse = np.sqrt(np.mean((x - y) ** 2))
    bias = np.mean(x - y)

    ax_cmp.text(
        0.02, 0.98,
        f"r = {r:.3f}\nRMSE = {rmse:.2f}\nBias = {bias:.2f}",
        transform=ax_cmp.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray")
    )

ax_cmp.set_title("Soil moisture vs SWC_F_MDS_1")
ax_cmp.set_xlabel("Modeled soil_moisture_mm")
ax_cmp.set_ylabel("Observed SWC_F_MDS_1")

# Hide unused axes
for j in range(n, len(axes)):
    axes[j].axis("off")

# Single legend for whole figure
handles, labels = ax_cmp.get_legend_handles_labels()
if labels:
    fig.legend(handles, labels, title="plant_type", loc="upper center", ncol=min(6, len(labels)))

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# COSMOS UK 

In [121]:
_D = pd.read_csv(
    "/home/vm/Desktop/cosmos-api-download-20260403140459/COSMOS_UK_RISEH_1D_202201010000_202512310000.csv",
    header=5,
    skiprows=[6, 7]
)
_D = _D.rename(columns={"parameter-id": "date"})
_D["date"] = pd.to_datetime(_D["date"], errors="coerce")
_D["date"]

0      2022-01-01 00:00:00+00:00
1      2022-01-02 00:00:00+00:00
2      2022-01-03 00:00:00+00:00
3      2022-01-04 00:00:00+00:00
4      2022-01-05 00:00:00+00:00
                  ...           
1456   2025-12-27 00:00:00+00:00
1457   2025-12-28 00:00:00+00:00
1458   2025-12-29 00:00:00+00:00
1459   2025-12-30 00:00:00+00:00
1460   2025-12-31 00:00:00+00:00
Name: date, Length: 1461, dtype: datetime64[us, UTC]

In [126]:
_L = pd.read_csv('/home/vm/Desktop/cosmos-uk_sitemetadata_2013-2023.csv')
land_cover_to_plant_type = {
    "Broadleaf woodland": "tree",
    "Coniferous woodland": "tree",
    "Coniferous forest": "tree",
    "Arable and horticulture": "crop",
    "Arable and horticulture (previously improved grassland)": "crop",
    "Orchard": "crop",
    "Grassland": "grass",
    "Improved grassland": "grass",
    "Heather grassland": "grass",
    "Acid grassland": "grass",
    "Calcareous grassland": "grass",
    "Heather": "shrub",
}

_L["plant_type"] = _L["LAND_COVER"].astype(str).str.strip().map(land_cover_to_plant_type)

# optional check
_L[["LAND_COVER", "plant_type"]].drop_duplicates().sort_values(["plant_type", "LAND_COVER"])

,LAND_COVER,plant_type
1,Arable and horticulture,crop
10,Arable and horticulture (previously improved g...,crop
37,Orchard,crop
22,Acid grassland,grass
26,Calcareous grassland,grass
2,Grassland,grass
7,Heather grassland,grass
4,Improved grassland,grass
17,Heather,shrub
0,Broadleaf woodland,tree


In [127]:
_L

,SITE_NAME,SITE_ID,START_DATE,END_DATE,EASTING,NORTHING,EAST_NORTH_PROJECTION,LATITUDE,LONGITUDE,LAT_LONG_PROJECTION,ALTITUDE,SOIL_TYPE,LAND_COVER,BULK_DENSITY,BULK_DENSITY_SD,SOIL_ORGANIC_CARBON,SOIL_ORGANIC_CARBON_SD,LATTICE_WATER,LATTICE_WATER_SD,plant_type
0,Alice Holt,ALIC1,2015-03-06,NaN,479950,139985,27700,51.153551,-0.858232,4326,74.0,Mineral soil,Broadleaf woodland,0.85,NaN,0.042,NaN,0.025,NaN,tree
1,Balruddery,BALRD,2014-05-15,NaN,331643,732797,27700,56.482297,-3.111488,4326,95.0,Mineral soil,Arable and horticulture,1.34,0.140,0.023,0.00076,0.018,0.00066,crop
2,Bickley Hall,BICKL,2015-01-28,NaN,353112,347903,27700,53.026350,-2.700530,4326,78.0,Mineral soil,Grassland,1.31,0.230,0.020,0.00360,0.010,0.00042,grass
3,Bunny Park,BUNNY,2015-01-27,NaN,458884,329606,27700,52.860730,-1.126850,4326,39.0,Mineral soil,Arable and horticulture,1.55,0.110,0.016,0.00150,0.008,0.00120,crop
4,Cardington,CARDT,2015-06-24,NaN,507991,246422,27700,52.105601,-0.424644,4326,29.0,Mineral soil,Improved grassland,1.14,0.170,0.040,0.00520,0.016,0.00180,grass
5,Cwm Garw,CGARW,2016-06-29,NaN,211346,231653,27700,51.951295,-4.746634,4326,283.0,Mineral soil,Improved grassland,0.96,0.260,0.048,0.00150,0.022,0.00086,grass
6,Chimney Meadows,CHIMN,2013-10-02,NaN,436113,201160,27700,51.708021,-1.478766,4326,66.0,Calcareous mineral soil,Improved grassland,1.37,0.095,0.027,0.00160,0.011,0.00180,grass
7,Chobham Common,CHOBH,2015-02-24,NaN,497731,164128,27700,51.367821,-0.597484,4326,39.0,Organic soil over mineral soil,Heather grassland,0.90,0.370,0.031,0.00840,0.003,0.00110,grass
8,Cochno,COCHN,2017-08-23,2020-11-16,249980,674651,27700,55.941421,-4.403543,4326,170.0,Mineral soil,Improved grassland,0.83,0.210,0.068,0.01400,0.019,0.00180,grass
9,Cockle Park,COCLP,2014-11-21,NaN,419544,591351,27700,55.216013,-1.694374,4326,84.0,Mineral soil,Arable and horticulture,1.21,0.084,0.033,0.00180,0.020,0.00150,crop
